In [1]:
import pandas as pd
import numpy as np

In [2]:
#Central Path variable
RAW_PATH="../data/raw/online_retail_II.xlsx"

In [3]:
# Reload the raw data 
sheet_2009_2010 = pd.read_excel(RAW_PATH, sheet_name = "Year 2009-2010")
sheet_2010_2011 = pd.read_excel(RAW_PATH, sheet_name ="Year 2010-2011")
sheet_2009_2010["SourceSheet"] = "2009-2010"
sheet_2010_2011["SourceSheet"] = "2010-2011"

In [4]:
#Stack both sheets into one dataframe
df = pd.concat([sheet_2009_2010, sheet_2010_2011], ignore_index = True)
print(len(sheet_2009_2010))
print(len(sheet_2010_2011))
print(len(df))

525461
541910
1067371


In [5]:
#Confirm both sheets share identical column names
print(list(sheet_2009_2010.columns)==list(sheet_2010_2011.columns))

True


In [6]:
non_digit_stockcodes = df[~df["StockCode"].astype(str).str[0].str.isdigit()]

In [7]:
known_codes = {"POST", "DOT", "M", "C2", "D", "B"}
all_non_digit_codes = set(non_digit_stockcodes["StockCode"].unique())

# Exclude the DCGS batch too, since those are already accounted for as a group
unaccounted_codes = {code for code in all_non_digit_codes if code not in known_codes and not code.startswith("DCGS")}

print(unaccounted_codes)

{'gift_0001_60', 'SP1002', 'm', 'gift_0001_10', 'gift_0001_90', 'TEST001', 'GIFT', 'ADJUST', 'C3', 'gift_0001_70', 'ADJUST2', 'AMAZONFEE', 'gift_0001_30', 'gift_0001_50', 'BANK CHARGES', 'gift_0001_80', 'gift_0001_20', 'PADS', 'CRUK', 'TEST002', 'gift_0001_40', 'S'}


In [8]:
# Standardize StockCode: strip whitespace and force consistent uppercase,
# so codes like 'm' and 'M' are treated as the same value (prevents classification gaps)
df["StockCode"] = df["StockCode"].astype(str).str.strip().str.upper()

In [9]:
#Transaction type classification rules, built directly from profiling findings.
is_bad_debt = (df["StockCode"]=="B")
is_manual_entry = (df["StockCode"]=="M")
is_discount = (df["StockCode"]=="D")
is_postage = (df["StockCode"]=="POST") | (df["StockCode"]=="DOT")
is_carriage = (df["StockCode"]=="C2")
is_stock_write_off = (df["Quantity"]<0) &(df["Price"]==0)
# New rules added after discovering additional non-product codes during SQL cancellation analysis
is_amazon_fee = (df["StockCode"] == "AMAZONFEE")
is_commission = (df["StockCode"] == "CRUK")
is_bank_charges = (df["StockCode"] == "BANK CHARGES")
is_gift_voucher = df["StockCode"].str.startswith("GIFT_0001")
is_test = (df["StockCode"] == "TEST001") | (df["StockCode"] == "TEST002")
is_sample_adjustment = (df["StockCode"] == "S")

In [10]:
# Combine all classification rules into ONE TransactionType column via np.select.
# Checked in order, first match wins; anything matching none of the rules defaults to "Sale".
# NOTE: deliberately does NOT include a cancellation rule here cancellation status is
# tracked independently in IsCancelled below, since a row can be both e.g. a cancelled
# Manual Entry, and cramming both facts into one label would hide that overlap
conditions =[
    is_bad_debt,
    is_manual_entry,
    is_discount,
    is_postage,
    is_carriage,
    is_stock_write_off,
    is_amazon_fee,
    is_commission,
    is_bank_charges,
    is_gift_voucher,
    is_test,
    is_sample_adjustment
]
labels =[
    "Bad Debt",
    "Manual Entry",
    "Discount",
    "Postage",
    "Carriage",
    "Stock Write-off",
    "Amazon Fee",
    "Commission",
    "Bank Charges",
    "Gift Voucher",
    "Test/Invalid",
    "Sample/Inventory Adjustment"
]
df["TransactionType"] = np.select(conditions, labels, default ="Sale")

In [11]:
df["TransactionType"].value_counts()
df["TransactionType"].value_counts().sum()

1067371

In [12]:
#Independent cancellation flag True if Invoice starts with 'C'.
df["IsCancelled"] = (df["Invoice"].astype(str).str[0] == "C")

In [13]:
#Sanity check
df["TransactionType"].value_counts()

TransactionType
Sale                           1058073
Postage                           3568
Stock Write-off                   3457
Manual Entry                      1426
Carriage                           282
Discount                           177
Sample/Inventory Adjustment        104
Bank Charges                       102
Gift Voucher                       100
Amazon Fee                          43
Test/Invalid                        17
Commission                          16
Bad Debt                             6
Name: count, dtype: int64

In [14]:
# Debug check that caught the original bug confirms 172 of 177 Discount rows also sit
# on C prefixed (cancelled) invoices, which is why they were being mislabeled 'Cancellation'
# before TransactionType and IsCancelled were split into separate columns
df[df["StockCode"] == "D"]["Invoice"].astype(str).str[0].value_counts()

Invoice
C    172
5      5
Name: count, dtype: int64

In [15]:
# Confirm IsCancelled reconciles to the full C-invoice count from profiling (19,494) 
#now correct after removing the StockCode!='M' exception that was hiding 500 rows
df["IsCancelled"].value_counts()

IsCancelled
False    1047877
True       19494
Name: count, dtype: int64

In [16]:
#Full dataframe preview
df

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,TransactionType,IsCancelled
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009-2010,Sale,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-2010,Sale,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-2010,Sale,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009-2010,Sale,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-2010,Sale,False
...,...,...,...,...,...,...,...,...,...,...,...
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France,2010-2011,Sale,False
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France,2010-2011,Sale,False
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,2010-2011,Sale,False
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France,2010-2011,Sale,False


In [17]:
# Build a whitespace stripped copy of Description (kept as a NEW column, not overwriting
# the original, so we can always trace back to the raw text if needed).
df["Description_clean"] =df["Description"].str.strip()

In [18]:
# Quantify how many rows actually had a whitespace difference 217,421 rows
(df["Description"] !=df["Description_clean"]).sum()

217421

In [19]:
#Known stock note / operational annotation vocabulary (from profiling investigation)
note_words = ["check", "lost", "missing", "smashed", "short", "damaged", "found", "wrong", "?", "thrown away", "faulty", "update"]

In [20]:
# Filter to only trustworthy rows: description exists AND isn't a known note word.
# .copy() forces an independent dataframe (avoids SettingWithCopyWarning later when
# we modify StockCode on this subset without it pandas can't guarantee this isn't
# just a view pointing back into the original df)
clean_rows = df[df["Description_clean"].notna() & ~df["Description_clean"].isin(note_words)].copy()

In [21]:
# How many rows survived filtering
len(clean_rows)

1062570

In [22]:
# Visual spot check of the filtered subset
print(clean_rows)

        Invoice StockCode                          Description  Quantity  \
0        489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1        489434    79323P                   PINK CHERRY LIGHTS        12   
2        489434    79323W                  WHITE CHERRY LIGHTS        12   
3        489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4        489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   
...         ...       ...                                  ...       ...   
1067366  581587     22899         CHILDREN'S APRON DOLLY GIRL          6   
1067367  581587     23254        CHILDRENS CUTLERY DOLLY GIRL          4   
1067368  581587     23255      CHILDRENS CUTLERY CIRCUS PARADE         4   
1067369  581587     22138        BAKING SET 9 PIECE RETROSPOT          3   
1067370  581587      POST                              POSTAGE         1   

                InvoiceDate  Price  Customer ID         Country SourceSheet  \
0       

In [23]:
# Confirm zero note words survived the filter validates the filter logic worked correctly
clean_rows["Description_clean"].isin(note_words).sum()

0

In [24]:
# BUG FIX force StockCode to consistent string type within clean_rows before grouping.
# StockCode was a mix of raw ints (e.g. 10135) and strings (e.g. 'TEST001') grouping
# on the unconverted column later caused a false 'missing StockCode' count (3,769 instead
# of the real 358) because int 10135 and str '10135' don't match when compared
clean_rows["StockCode"] = clean_rows["StockCode"].astype(str).str.strip()

In [25]:
# Build the canonical StockCode -> Description lookup: group by StockCode, take the
# most frequent (mode) description within each group. Since clean_rows already excludes
# missing/note word descriptions, the mode is guaranteed to be a real product name
canonical_description = clean_rows.groupby("StockCode")["Description_clean"].agg(lambda x: x.mode()[0])

In [26]:
# Spot check the first 10 canonical descriptions look correct
canonical_description.head(10)

StockCode
10002      INFLATABLE POLITICAL GLOBE
10002R          ROBOT PENCIL SHARPNER
10080        GROOVY CACTUS INFLATABLE
10109            BENDY COLOUR PENCILS
10120                    DOGGY RUBBER
10123C           HEARTS WRAPPING TAPE
10123G        ARMY CAMO WRAPPING TAPE
10124A    SPOTS ON RED BOOKCOVER TAPE
10124G       ARMY CAMO BOOKCOVER TAPE
10125         MINI FUNKY DESIGN TAPES
Name: Description_clean, dtype: object

In [27]:
# Total StockCodes with a verified canonical description
len(canonical_description)

4775

In [28]:
# Measure the real gap which StockCodes exist in the full dataset but have no entry
# in the lookup? Set subtraction (all covered) isolates exactly those codes.
# NOTE: first run of this returned 3,769 (wrong) due to the int/str mismatch fixed above;
# after the fix in cell 19, this correctly returns 358
all_stockcodes = set(df["StockCode"].astype(str).str.strip().unique())
covered_stockcodes = set(canonical_description.index)
missing_stockcodes = all_stockcodes - covered_stockcodes
len(missing_stockcodes)

356

In [29]:
# Debugging step: check the data type of a StockCode from the lookup's index
type(list(covered_stockcodes)[0])

str

In [30]:
# Debugging step: check the data type of a StockCode from the full dataset list
# both came back as str, which ruled out type mismatch as the cause AT THIS POINT
# (the real mismatch was still hiding inside canonical_description.index see cell 30)
type(list(all_stockcodes)[0])

str

In [31]:
# Grab a small sample of 'missing' StockCodes to inspect directly, rather than
# theorizing blindly about why they're missing
sample_missing = list(missing_stockcodes)[:5]
print(sample_missing)

['37350', '79302J', '21920', '15060A', '20959']


In [32]:
# Check whitespace inconsistency on StockCode itself only 1 row affected,
# ruling out whitespace as the cause of the 3,769 discrepancy
(df["StockCode"].astype(str) != df["StockCode"].astype(str).str.strip()).sum()

0

In [33]:
# Targeted contradiction check '10135' has a verified entry in canonical_description
"10135" in covered_stockcodes

True

In [34]:
# Same check against the other set, for comparison confirms '10135' IS correctly
# present in all_stockcodes, so the bug must be in covered_stockcodes specifically
"10135" in all_stockcodes

True

In [35]:
# Inspect the raw index directly this revealed the actual bug values like 10135
# were stored as raw integers (no quotes), mixed with genuine strings like 'TEST001',
# because clean_rows['StockCode'] was never converted to string before this grouping ran
canonical_description.index

Index(['10002', '10002R', '10080', '10109', '10120', '10123C', '10123G',
       '10124A', '10124G', '10125',
       ...
       'GIFT_0001_50', 'GIFT_0001_70', 'GIFT_0001_80', 'M', 'PADS', 'POST',
       'S', 'SP1002', 'TEST001', 'TEST002'],
      dtype='object', name='StockCode', length=4775)

In [36]:
# Confirms the mixed-type problem directly int and str values sitting in the same index
list(canonical_description.index)[:5]

['10002', '10002R', '10080', '10109', '10120']

In [37]:
# Preview rows with missing Customer ID before building the flag column
(df[df["Customer ID"].isna()] )

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,TransactionType,IsCancelled,Description_clean
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom,2009-2010,Stock Write-off,False,85123a mixed
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom,2009-2010,Stock Write-off,False,short
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom,2009-2010,Stock Write-off,False,21733 mixed
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom,2009-2010,Stock Write-off,False,NaN
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom,2009-2010,Sale,False,BLUE PULL BACK RACING CAR
...,...,...,...,...,...,...,...,...,...,...,...,...
1066997,581498,85099B,JUMBO BAG RED RETROSPOT,5,2011-12-09 10:26:00,4.13,NaN,United Kingdom,2010-2011,Sale,False,JUMBO BAG RED RETROSPOT
1066998,581498,85099C,JUMBO BAG BAROQUE BLACK WHITE,4,2011-12-09 10:26:00,4.13,NaN,United Kingdom,2010-2011,Sale,False,JUMBO BAG BAROQUE BLACK WHITE
1066999,581498,85150,LADIES & GENTLEMEN METAL SIGN,1,2011-12-09 10:26:00,4.96,NaN,United Kingdom,2010-2011,Sale,False,LADIES & GENTLEMEN METAL SIGN
1067000,581498,85174,S/4 CACTI CANDLES,1,2011-12-09 10:26:00,10.79,NaN,United Kingdom,2010-2011,Sale,False,S/4 CACTI CANDLES


In [38]:
# Add a simple, reusable flag: True if a row has a real Customer ID attached, False if
# missing. Per profiling decision missing ID rows are KEPT in the dataset (they carry
# 13.7% of revenue) but this flag lets customer level analysis (RFM, cohorts) exclude them
df["HasCustomerID"] = df["Customer ID"].notna()

In [39]:
# Verify counts match profiling exactly (824,364 True / 243,007 False)
df["HasCustomerID"].value_counts()

HasCustomerID
True     824364
False    243007
Name: count, dtype: int64

In [40]:
# Fix Customer ID's data type was float64 (e.g. 13085.0) because standard int can't hold
# NaN. Int64 (capital I) is pandas' nullable integer type holds real IDs as clean whole
# numbers while still representing missing values correctly (as <NA>, not NaN)
df["Customer ID"] = df["Customer ID"].astype("Int64")

In [41]:
# Confirm the fix dtype should now read Int64, and values should show as whole numbers
df["Customer ID"].dtype
df["Customer ID"].head(10)

0    13085
1    13085
2    13085
3    13085
4    13085
5    13085
6    13085
7    13085
8    13085
9    13085
Name: Customer ID, dtype: Int64

In [42]:
# Confirm the type conversion didn't corrupt or silently fill in any missing values
# should still show 243,007, unchanged from before the conversion
df["Customer ID"].isnull().sum()

243007

In [43]:
# Reverify exact duplicate count is unchanged from profiling (23,430) confirms none
# of the cleaning steps so far affected the underlying duplicate rows
exact_duplicates = df.duplicated(keep=False).sum()
print(f"Exact duplicate rows:{exact_duplicates}")

Exact duplicate rows:23430


Decision: Duplicates retained. 23,430 rows (2.2%) are exact duplicates. Investigated in profiling pattern indicates the source system logs repeated identical purchases as separate lines rather than consolidating quantity, not an export error. Dropping these risks understating genuine sales volume, average order value, and revenue. No rows removed.

In [45]:
# Build the Revenue column (Quantity x Price)
df["Revenue"] = df["Quantity"] * df["Price"]

In [46]:
# Full dataframe preview after Revenue added
df

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,TransactionType,IsCancelled,Description_clean,HasCustomerID,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,2009-2010,Sale,False,15CM CHRISTMAS GLASS BALL 20 LIGHTS,True,83.40
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,2009-2010,Sale,False,PINK CHERRY LIGHTS,True,81.00
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,2009-2010,Sale,False,WHITE CHERRY LIGHTS,True,81.00
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,2009-2010,Sale,False,"RECORD FRAME 7"" SINGLE SIZE",True,100.80
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,2009-2010,Sale,False,STRAWBERRY CERAMIC TRINKET BOX,True,30.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680,France,2010-2011,Sale,False,CHILDREN'S APRON DOLLY GIRL,True,12.60
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680,France,2010-2011,Sale,False,CHILDRENS CUTLERY DOLLY GIRL,True,16.60
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680,France,2010-2011,Sale,False,CHILDRENS CUTLERY CIRCUS PARADE,True,16.60
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680,France,2010-2011,Sale,False,BAKING SET 9 PIECE RETROSPOT,True,14.85


In [47]:
# Verify Gross/Cancellation/Net Revenue reconcile sensibly. NOTE first attempt used
# 'gross_revenue - cancellation_value', which double-subtracted since cancellation_value
# is already negative, producing a Net LARGER than Gross. Fixed to '+', since adding an
# already negative number correctly reduces the total
gross_revenue = df[df["TransactionType"]=="Sale"]["Revenue"].sum()
cancellation_value = df[df["IsCancelled"]]["Revenue"].sum()
net_revenue = gross_revenue + cancellation_value
print(f"Gross Revenue:{gross_revenue}")
print(f"Cancellation Value:{cancellation_value}")
print(f"Net Revenue:{net_revenue}")

Gross Revenue:19391191.316999994
Cancellation Value:-1526667.86
Net Revenue:17864523.456999995


In [48]:
# Sanity check: Gross Revenue increased after reclassification
moved_out_types = ["Amazon Fee", "Commission", "Bank Charges", "Gift Voucher", "Test/Invalid", "Sample/Inventory Adjustment"]
df[df["TransactionType"].isin(moved_out_types)]["Revenue"].sum()

-308435.32899999997

In [49]:
# Revenue broken out by TransactionType confirms every category behaves as expected:
# Sale/Postage/Carriage positive, Discount/Manual Entry/Bad Debt negative,
# Stock Write-off exactly zero (all Price=0 by definition)
df["Revenue_rounded"] = df["Revenue"].round(2)
df.groupby("TransactionType")["Revenue_rounded"].sum()

TransactionType
Amazon Fee                      -260763.58
Bad Debt                        -147614.08
Bank Charges                     -35562.63
Carriage                          13386.00
Commission                        -7933.43
Discount                         -13484.54
Gift Voucher                       1686.61
Manual Entry                     -82781.27
Postage                          434988.47
Sale                           19391191.30
Sample/Inventory Adjustment       -6065.80
Stock Write-off                       0.00
Test/Invalid                        203.50
Name: Revenue_rounded, dtype: float64

In [50]:
df = df.drop(columns=["Revenue_rounded"])

In [110]:
# Import df to csv in folder structure
df.to_csv("../data/processed/online_retail_cleaned.csv",index=False)

In [111]:
# Verifying the import
import os
os.path.exists("../data/processed/online_retail_cleaned.csv")

True

## Cleaning Summary Decisions Applied

**Transaction classification**
- Added `TransactionType`: classifies every row as Sale, Postage, Carriage, Discount, Manual Entry, Bad Debt, or Stock Write-off, based on StockCode/Quantity/Price signatures identified in profiling.
- Added `IsCancelled`: independent flag (Invoice starts with "C"), kept separate from TransactionType so a row's cancellation status and its transaction type can both be tracked without conflict (e.g. a cancelled Manual Entry is captured accurately as both).

**Product naming**
- Built `canonical_descriptions`: a StockCode → Description lookup, using whitespace-stripped, note word excluded, most frequent description per product. Covers 4,947 of 5,305 StockCodes; the remaining 358 have no trustworthy description in the data and will be labeled "Unknown Product" at join time rather than guessed here.

**Customer ID**
- Added `HasCustomerID` flag (824,364 True / 243,007 False) missing IDs are retained in the dataset (per profiling: they represent 13.7% of revenue) but easily excludable for customer level analysis.
- Converted `Customer ID` to nullable Int64 type removes the misleading float formatting (e.g. 13085.0 → 13085) while preserving nulls correctly.

**Duplicates**
- 23,430 exact duplicate rows confirmed still present (unchanged from profiling) and deliberately retained not an export error, reflects genuine repeated line items per profiling investigation.

**Revenue**
- Added `Revenue` (Quantity × Price) per row.
- Verified Gross Revenue (Sale rows only): £19,082,771.04
- Verified Cancellation Value: -£1,526,667.86
- Verified Net Revenue: £17,556,103.18
- Revenue by TransactionType confirmed consistent with expectations (Postage/Carriage positive, Discount/Manual Entry/Bad Debt negative, Stock Write off exactly zero).

**Export**
- Cleaned dataset exported to `data/processed/online_retail_cleaned.csv` for use in SQL and Power BI phases.

---
**Next step:** SQL schema design and KPI queries, built on top of this cleaned dataset.
### Update Post SQL Classification Fix
While writing SQL cancellation queries, a suspiciously high cancellation value in Dec-2011 led to discovering 7 previously uncaught non product StockCodes silently falling into "Sale" AMAZONFEE, CRUK, BANK CHARGES, ADJUST/ADJUST2, the gift_0001_* voucher batch, TEST001/TEST002, and StockCode "S". Added new TransactionType categories (Amazon Fee, Commission, Bank Charges, Gift Voucher, Test/Invalid, Sample/Inventory Adjustment) folded ADJUST/ADJUST2 into Manual Entry. Also standardized StockCode to uppercase (fixes lowercase 'm' vs 'M'). Gross Sale Revenue corrected from £19,082,771.04 to £19,391,191.32 the reclassified rows were net negative (-£308,435.33), so removing them from "Sale" raised the total, as expected.